# Market Gap Analysis

In [8]:
import pandas as pd
import numpy as np
import re
from collections import Counter

In [9]:
!apt-get -qq install -y aria2
!aria2c -x 16 -s 16 -o off.csv.gz "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
!ls -lh off.csv.gz

Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.11) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [10]:
COLUMNS = ["product_name","categories_tags","sugars_100g","proteins_100g","ingredients_text"]

TARGET_ROWS = 60_000
CHUNKSIZE = 10_000

chunks = []
total = 0

reader = pd.read_csv(
    "off.csv.gz",
    sep="\t",
    usecols=COLUMNS,
    chunksize=CHUNKSIZE,
    low_memory=True,
    on_bad_lines="skip"
)

for c in reader:
    chunks.append(c)
    total += len(c)
    if total >= TARGET_ROWS:
        break

df_raw = pd.concat(chunks, ignore_index=True)
df_raw.shape

(60000, 5)

In [11]:
df = df_raw.copy()

df["sugars_100g"] = pd.to_numeric(df["sugars_100g"], errors="coerce")
df["proteins_100g"] = pd.to_numeric(df["proteins_100g"], errors="coerce")

# Drop missing critical fields
df = df.dropna(subset=["product_name","sugars_100g","proteins_100g"])

# Remove biologically impossible values
df = df[
    df["sugars_100g"].between(0, 100) &
    df["proteins_100g"].between(0, 100)
].copy()

df.shape

(1174, 5)

In [12]:
df["first_category"] = df["categories_tags"].astype("string").str.split(",").str[0]
df["first_category"].value_counts().head(20)

,count
first_category,
en:plant-based-foods-and-beverages,433
en:dairies,404
en:undefined,22
en:snacks,20
en:condiments,8
en:beverages-and-beverages-preparations,8
en:null,8
en:meats-and-their-products,7
en:seafood,6


In [13]:
def map_first_category(cat):
    if pd.isna(cat):
        return "Other"
    cat = str(cat).lower()

    if "snacks" in cat:
        return "Snacks"
    if "desserts" in cat:
        return "Desserts"
    if "beverages" in cat:
        return "Beverages"
    if "dairies" in cat:
        return "Dairy"
    if "condiments" in cat or "sauces" in cat:
        return "Condiments & Sauces"
    if "frozen" in cat:
        return "Frozen Foods"
    if "breakfast" in cat or "cereals" in cat:
        return "Cereals & Breakfast"
    if "plant-based" in cat:
        return "Plant-Based Foods"
    if "meals" in cat:
        return "Prepared Meals"
    if "meats" in cat or "seafood" in cat:
        return "Meat & Seafood"
    if cat in ["en:undefined", "en:null"]:
        return "Other"

    return "Other"

df["primary_category"] = df["first_category"].apply(map_first_category)
df["primary_category"].value_counts()

,count
primary_category,
Beverages,444
Dairy,404
Other,279
Snacks,22
Meat & Seafood,13
Condiments & Sauces,8
Prepared Meals,2
Frozen Foods,1
Desserts,1


In [14]:
HP_MIN = 10
LS_MAX = 5

df["is_opportunity"] = (df["proteins_100g"] > HP_MIN) & (df["sugars_100g"] < LS_MAX)
blue_ocean = df[df["is_opportunity"]].copy()

blue_ocean["primary_category"].value_counts()

,count
primary_category,
Other,71
Beverages,69
Dairy,13
Meat & Seafood,12
Frozen Foods,1
Snacks,1
Condiments & Sauces,1


In [15]:
total_counts = df["primary_category"].value_counts()
blue_counts = blue_ocean["primary_category"].value_counts()

analysis = pd.DataFrame({
    "Total Products": total_counts,
    "HighProtein_LowSugar": blue_counts
}).fillna(0)

analysis["Opportunity_Ratio"] = analysis["HighProtein_LowSugar"] / analysis["Total Products"]
analysis.sort_values("Opportunity_Ratio", ascending=True)

,Total Products,HighProtein_LowSugar,Opportunity_Ratio
primary_category,,,
Desserts,1,0.0,0.000000
Prepared Meals,2,0.0,0.000000
Dairy,404,13.0,0.032178
Snacks,22,1.0,0.045455
Condiments & Sauces,8,1.0,0.125000
Beverages,444,69.0,0.155405
Other,279,71.0,0.254480
Meat & Seafood,13,12.0,0.923077
Frozen Foods,1,1.0,1.000000


In [18]:
snack_blue = blue_ocean[blue_ocean["primary_category"] == "Snacks"]
snack_blue[["proteins_100g","sugars_100g"]].describe(percentiles=[0.25,0.5,0.75])

,proteins_100g,sugars_100g
count,1.0,1.000
mean,12.5,3.125
std,NaN,NaN
min,12.5,3.125
25%,12.5,3.125
50%,12.5,3.125
75%,12.5,3.125
max,12.5,3.125


In [19]:
kpi_table = (
    df.groupby("primary_category")
      .agg(
          total_products=("product_name", "count"),
          opportunity_products=("is_opportunity", "sum"),
          opportunity_density=("is_opportunity", "mean"),
      )
      .reset_index()
)

kpi_table["opportunity_density"] = (kpi_table["opportunity_density"] * 100).round(2)
kpi_table.sort_values("opportunity_density", ascending=False).head(10)

,primary_category,total_products,opportunity_products,opportunity_density
4,Frozen Foods,1,1,100.00
5,Meat & Seafood,13,12,92.31
6,Other,279,71,25.45
0,Beverages,444,69,15.54
1,Condiments & Sauces,8,1,12.50
8,Snacks,22,1,4.55
2,Dairy,404,13,3.22
3,Desserts,1,0,0.00
7,Prepared Meals,2,0,0.00


In [29]:
kpi_table.to_csv("kpi_opportunity_density.csv", index=False)

In [30]:
!ls -lh kpi_opportunity_density.csv

-rw-r--r-- 1 root root 269 Mar  1 08:44 kpi_opportunity_density.csv


In [31]:
from google.colab import files
files.download("kpi_opportunity_density.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
good = df[df["is_opportunity"] & df["ingredients_text"].notna()].copy()
text = good["ingredients_text"].str.lower().fillna("")

protein_sources = [
    "whey", "milk protein", "casein",
    "soy", "soya",
    "pea protein", "pea",
    "peanut", "groundnut",
    "almond", "nuts",
    "egg", "egg white",
    "chickpea", "lentil", "beans",
    "oat", "quinoa",
    "collagen", "gelatin"
]

counts = {}
for s in protein_sources:
    counts[s] = int(text.str.contains(r"\b" + re.escape(s) + r"\b", regex=True).sum())

top3 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]
top3

[('soy', 21), ('whey', 15), ('oat', 5)]

In [23]:
hidden_gem_df = pd.DataFrame(top3, columns=["protein_source","count"])
hidden_gem_df.to_csv("hidden_gem_top_protein_sources.csv", index=False)

In [24]:
!ls -lh hidden_gem_top_protein_sources.csv

-rw-r--r-- 1 root root 42 Mar  1 08:35 hidden_gem_top_protein_sources.csv


In [25]:
from google.colab import files
files.download("hidden_gem_top_protein_sources.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>